# XAUUSD Scalping ML Model - Kaggle Training

This notebook trains an advanced ML model for XAUUSD (Gold/USD) scalping signal generation.

**Features:**
- Advanced feature engineering with 50+ technical indicators
- Multiple ML models (XGBoost, LightGBM, Random Forest)
- Time-series cross-validation
- Hyperparameter tuning
- Model comparison and selection

**Output:** Trained model saved to `/kaggle/working/models/`

## 1. Setup and Imports

In [ ]:
# Check if running on Kaggle
import os
import json
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

# Install required packages (if needed)
!pip install -q pandas-ta ta xgboost lightgbm joblib

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from pathlib import Path
from datetime import datetime

import yfinance as yf
import ta

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Setup complete")

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    'symbol': 'GC=F',
    'period': '5y',
    'interval': '1h',
    'lookahead': 5,
    'test_size': 0.2,
    'random_state': 42
}

# Paths
if IS_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working')
    MODEL_DIR = OUTPUT_DIR / 'models'
else:
    OUTPUT_DIR = Path('output')
    MODEL_DIR = OUTPUT_DIR / 'models'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {MODEL_DIR}")

## 3. Data Fetching

In [ ]:
def fetch_xauusd_data(period='730d', interval='1h'):
    """Fetch XAUUSD historical data from Yahoo Finance."""
    print(f"Fetching {CONFIG['symbol']} data: period={period}, interval={interval}")
    ticker = yf.Ticker(CONFIG['symbol'])
    df = ticker.history(period=period, interval=interval)
    if df.empty:
        raise ValueError('No data fetched!')
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    print(f"Fetched {len(df)} rows")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    return df

df_raw = fetch_xauusd_data()
df_raw.head()

## 4. Feature Engineering

In [ ]:
class FeatureEngineer:
    def create_all_features(self, df):
        df = df.copy()
        df.columns = [c.lower() for c in df.columns]
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
        df['high_low_range'] = df['high'] - df['low']
        df['body_size'] = abs(df['close'] - df['open'])
        df['upper_shadow'] = df['high'] - df[['close', 'open']].max(axis=1)
        df['lower_shadow'] = df[['close', 'open']].min(axis=1) - df['low']
        df['direction'] = (df['close'] > df['open']).astype(int)
        df['rsi_14'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
        df['rsi_7'] = ta.momentum.RSIIndicator(df['close'], window=7).rsi()
        macd = ta.trend.MACD(df['close'], window_fast=12, window_slow=26, window_sign=9)
        if macd is not None:
            df['macd'] = macd.macd()
            df['macd_signal'] = macd.macd_signal()
            df['macd_hist'] = macd.macd_diff()
        stoch = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close'], window=14, smooth_window=3)
        if stoch is not None:
            df['stoch_k'] = stoch.stoch()
            df['stoch_d'] = stoch.stoch_signal()
        df['williams_r'] = ta.momentum.WilliamsRIndicator(df['high'], df['low'], df['close'], lbp=14).williams_r()
        df['cci_20'] = ta.trend.CCIIndicator(df['high'], df['low'], df['close'], window=20).cci()
        df['momentum_10'] = df['close'] - df['close'].shift(10)
        df['sma_10'] = df['close'].rolling(window=10).mean()
        df['sma_20'] = df['close'].rolling(window=20).mean()
        df['sma_50'] = df['close'].rolling(window=50).mean()
        df['ema_10'] = df['close'].ewm(span=10, adjust=False).mean()
        df['ema_20'] = df['close'].ewm(span=20, adjust=False).mean()
        df['ema_50'] = df['close'].ewm(span=50, adjust=False).mean()
        df['price_vs_ema20'] = (df['close'] - df['ema_20']) / df['ema_20']
        df['price_vs_ema50'] = (df['close'] - df['ema_50']) / df['ema_50']
        adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        if adx is not None:
            df['adx'] = adx.adx()
            df['adx_pos'] = adx.adx_pos()
            df['adx_neg'] = adx.adx_neg()
        df['ema_cross_signal'] = np.where(
            df['ema_10'] > df['ema_20'], 1,
            np.where(df['ema_10'] < df['ema_20'], -1, 0))
        atr = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=14).average_true_range()
        if atr is not None:
            df['atr_14'] = atr
        df['atr_7'] = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=7).average_true_range()
        df['atr_ratio'] = df['atr_14'] / df['atr_14'].rolling(50).mean()
        bbands = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        if bbands is not None:
            df['bollinger_upper'] = bbands.bollinger_hband()
            df['bollinger_middle'] = bbands.bollinger_mavg()
            df['bollinger_lower'] = bbands.bollinger_lband()
            df['bollinger_width'] = (df['bollinger_upper'] - df['bollinger_lower']) / df['bollinger_middle']
            df['bollinger_pct'] = (df['close'] - df['bollinger_lower']) / (df['bollinger_upper'] - df['bollinger_lower'])
        df['volume_sma_20'] = df['volume'].rolling(window=20).mean()
        df['volume_ratio'] = df['volume'] / df['volume_sma_20']
        df['obv'] = ta.volume.OnBalanceVolumeIndicator(df['close'], df['volume']).on_balance_volume()
        df['cmf_20'] = ta.volume.ChaikinMoneyFlowIndicator(df['high'], df['low'], df['close'], df['volume'], window=20).chaikin_money_flow()
        for lag in [1, 2, 3, 5, 10]:
            df[f'close_lag_{lag}'] = df['close'].shift(lag)
            df[f'returns_lag_{lag}'] = df['returns'].shift(lag)
            df[f'rsi_lag_{lag}'] = df['rsi_14'].shift(lag)
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['month'] = df.index.month
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['is_london_ny_overlap'] = ((df['hour'] >= 13) & (df['hour'] < 16)).astype(int)
        return df
    def create_target(self, df, lookahead=5):
        future_returns = df['close'].shift(-lookahead) / df['close'] - 1
        df['target'] = (future_returns > 0).astype(int)
        df = df.dropna(subset=['target'])
        return df

fe = FeatureEngineer()
df_features = fe.create_all_features(df_raw)
df_features = fe.create_target(df_features, lookahead=CONFIG['lookahead'])
df_features = df_features.dropna()
print(f"Feature engineering complete!")
print(f"Shape: {df_features.shape}")
print(f"Target distribution:")
print(df_features['target'].value_counts(normalize=True))

## 5. Feature Correlation Analysis

In [ ]:
exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'target']
feature_cols = [c for c in df_features.columns if c not in exclude_cols]
correlations = df_features[feature_cols + ['target']].corr()['target'].drop('target')
correlations = correlations.abs().sort_values(ascending=False)
plt.figure(figsize=(12, 10))
correlations.head(30).plot(kind='barh')
plt.title('Top 30 Feature Correlations with Target')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()
print(f"Total features: {len(feature_cols)}")
print('\nTop 10 correlated features:')
print(correlations.head(10))

## 6. Train-Test Split

In [ ]:
X = df_features[feature_cols].values
y = df_features['target'].values
split_idx = int(len(df_features) * (1 - CONFIG['test_size']))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Train target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

## 7. Model Training & Comparison

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
scale_pos_weight = class_weights[1] / class_weights[0]

models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic', eval_metric='logloss',
        random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective='binary', random_state=42, n_jobs=-1, verbosity=-1),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_split=5, min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1)
}

results = {}
trained_models = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print('='*50)
    model.fit(X_train_scaled, y_train)
    train_preds = model.predict(X_train_scaled)
    test_preds = model.predict(X_test_scaled)
    test_proba = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'train_accuracy': accuracy_score(y_train, train_preds),
        'test_accuracy': accuracy_score(y_test, test_preds),
        'test_precision': precision_score(y_test, test_preds, zero_division=0),
        'test_recall': recall_score(y_test, test_preds, zero_division=0),
        'test_f1': f1_score(y_test, test_preds, zero_division=0),
        'test_auc': roc_auc_score(y_test, test_proba)}
    trained_models[name] = model
    print(f"Train Accuracy: {results[name]['train_accuracy']:.4f}")
    print(f"Test Accuracy: {results[name]['test_accuracy']:.4f}")
    print(f"Test F1: {results[name]['test_f1']:.4f}")
    print(f"Test AUC: {results[name]['test_auc']:.4f}")
    print('\nClassification Report:')
    print(classification_report(y_test, test_preds, target_names=['DOWN', 'UP']))
print('\n✓ All models trained!')

## 8. Model Comparison Visualization

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.round(4))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_auc']
titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']
for i, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[i // 3, i % 3]
    results_df[metric].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
best_model_name = results_df['test_f1'].idxmax()
best_model = trained_models[best_model_name]
ax = axes[1, 2]
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(
        best_model.feature_importances_,
        index=feature_cols).sort_values(ascending=True).tail(15)
    importance.plot(kind='barh', ax=ax)
    ax.set_title(f'Top 15 Features ({best_model_name})')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBest model: {best_model_name}")

## 9. Cross-Validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(
    best_model, X_train_scaled, y_train,
    cv=tscv, scoring='f1', n_jobs=-1)
print(f"5-Fold Time-Series CV F1 Scores: {cv_scores}")
print(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), cv_scores, marker='o', linewidth=2, markersize=8)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.title(f'5-Fold Time-Series CV ({best_model_name})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Confusion Matrix

In [ ]:
test_preds = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['DOWN', 'UP'],
            yticklabels=['DOWN', 'UP'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## 11. Save Model

In [ ]:
model_filename = 'xauusd_best_model'
model_path = MODEL_DIR / f'{model_filename}.joblib'
scaler_path = MODEL_DIR / f'{model_filename}_scaler.joblib'
features_path = MODEL_DIR / f'{model_filename}_features.txt'
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
with open(features_path, 'w') as f:
    f.write('\n'.join(feature_cols))
print(f"✓ Model saved to: {model_path}")
print(f"✓ Scaler saved to: {scaler_path}")
print(f"✓ Features saved to: {features_path}")
summary = {
    'model_type': best_model_name,
    'training_date': datetime.now().isoformat(),
    'data_period': CONFIG['period'],
    'interval': CONFIG['interval'],
    'lookahead': CONFIG['lookahead'],
    'n_features': len(feature_cols),
    'n_samples': len(df_features),
    'metrics': results[best_model_name],
    'cv_f1_mean': float(cv_scores.mean()),
    'cv_f1_std': float(cv_scores.std())}
summary_path = MODEL_DIR / 'training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✓ Training summary saved to: {summary_path}")
print('\n' + '='*50)
print('TRAINING COMPLETE!')
print('='*50)
print(f"Best Model: {best_model_name}")
print(f"Test Accuracy: {results[best_model_name]['test_accuracy']:.4f}")
print(f"Test F1: {results[best_model_name]['test_f1']:.4f}")
print(f"Test AUC: {results[best_model_name]['test_auc']:.4f}")
print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 12. Download Model Files

In [ ]:
print('Output files:')
for f in MODEL_DIR.glob('*'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.2f} MB")
if IS_KAGGLE:
    from IPython.display import FileLink
    for f in MODEL_DIR.glob('*'):
        display(FileLink(str(f)))
    print('\n✓ Click the links above to download model files')